# F2 · Madde 3 — Her saldırı ailesi sırayla dışarıda (leave-one-family-out, LOFO)
**Profesörün notu:** *Her saldırı ailesini sırayla dışarıda bırakarak unseen-attack değerlendirmesini gerçekleştir.*

Bu protokol öneri v6 §7a.3'te (E7) **ilk doğrulayıcı çalıştırmadan önce** kaydedildi:
* 9 katman (fold): her saldırı ailesi bir kez görülmemiş; diğer 8 aile bilinen (geliştirme) aile;
* **yeni eğitim tohumları 10–19**; veri bölme tohumu değişmez; AE boyutu 32-16-8 sabit (katman başına seçim yok);
* karşılaştırma kümesi: AE (üç karar, akış düzeyi) · AE + Basamak 2 (host sayacı) · AE + Basamak 1 + Basamak 2 ·
  **yalnız RF** (8 bilinen aileyle eğitilir) eşleştirilmiş yanlış alarmda;
* W ∈ {20, 50, 100}, F ∈ {1.000, 5.000}, ρ ∈ {0,10; 0,25; 0,50; 1,00}. Sürüm B'de zaman damgası olmadığı için host
  pencereleri **simülasyondur**.

**Başarı ölçütü (§7a.3):** AE + Basamak 2, ρ = 1,0'da görülmemiş ailenin kaçan + ertelenen oranını **en az 30 puan**
azaltmalı ve en kötü hostun yanlış yükseltmesi bütçe içinde (haftada ≤ 1) kalmalı; bu **9 katmanın en az 6'sında** sağlanmalı.
**Başarısızlık ölçütü:** bilinenlere *benzemeyen* ailelerde (bu katmanda yalnız-RF recall < 0,50) yalnız RF, AE + Basamak 2'ye
yetişirse (≥ AE + Basamak 2 − 0,05), "yalnız normal veriyle" iddiası TON_IoT için geri çekilir ve veri kümesi bulgusu olarak raporlanır.
(< 0,50 ve 0,05 eşikleri öneride açık bırakılmıştı; bu defter çalıştırılmadan önce `configs/default.json`'da sabitlendi.)

**Neden AE her katmanda yeniden eğitilmiyor?** AE yalnız normal-eğitim verisini görür ve normal bölme saldırı aileleri
atanmadan önce çekilir; normal veri her katmanda aynıdır. Defter bunu varsaymaz, **assert** ile doğrular.
**Öneriden sapma (açıkça):** §7a.3 istemci başına (Mondrian) k\* kalibrasyonu yazar; bu merkezi LOFO'da istemci olmadığı için
k\* ara dönemdeki gibi normal-doğrulama pencerelerinden hesaplanır.

In [ ]:
# ---- Kurulum (her defterde aynı): Drive'ı bağla, ortak kodu içe aktar ----
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
PROJECT_DIR = os.environ.get("TRUSTIDS_PROJECT", "/content/drive/MyDrive/TrustFed-TinyIDS-Final")
QUICK = os.environ.get("TRUSTIDS_QUICK", "0") == "1"   # hızlı mod YALNIZCA kodu test etmek içindir
# Devam modu: Colab oturumu kapanırsa, Drive'a kaydedilmiş tohumlar/adımlar atlanır ve kalan yerden devam edilir.
# Her şeyi baştan hesaplamak için: os.environ["TRUSTIDS_RESUME"] = "0"
RESUME = os.environ.get("TRUSTIDS_RESUME", "1") == "1"
sys.path.insert(0, PROJECT_DIR)

from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from trustids.config import Config
from trustids import utils

# Veri kümesi seçimi: varsayılan TON_IoT; ikinci veri kümesi için TRUSTIDS_CONFIG=edge_iiotset.json
CONFIG_NAME = os.environ.get("TRUSTIDS_CONFIG", "default.json")
CFG_PATH = Path(PROJECT_DIR) / "configs" / CONFIG_NAME
if CONFIG_NAME != "default.json" and not CFG_PATH.exists():
    raise FileNotFoundError(f"{CFG_PATH} bulunamadı: configs/ klasörünü de Drive'a yükleyin.")
cfg = Config.load(CFG_PATH) if CFG_PATH.exists() else Config(project_dir=PROJECT_DIR)
cfg.project_dir = PROJECT_DIR
if QUICK:
    cfg.train_seeds, cfg.validity_resplits, cfg.epochs = cfg.train_seeds[:2], 30, 15
    cfg.random_control_repeats, cfg.bootstrap_repeats = 10, 200
    cfg.lofo_seeds, cfg.fl_orig_rounds = cfg.lofo_seeds[:2], 3
    cfg.resolution_cal_windows, cfg.resolution_eval_windows = 600, 600
cfg.check()
ART = cfg.artifacts
FIG = Path(PROJECT_DIR) / cfg.figures_dir; FIG.mkdir(exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("Proje klasörü:", PROJECT_DIR, "| hızlı mod:", QUICK, "| devam modu:", RESUME)
print("Drive'da kayıtlı ilerleme:", utils.progress(cfg))

In [ ]:
import tensorflow as tf
from trustids import data as D, models as M, conformal as C, resolution as R, lofo as L
raw = D.load_raw(cfg.data_path, cfg)
P0 = D.load_prepared(ART / "prepared")
spec = cfg.ae_variants[cfg.ae_default]            # önceden kaydedildiği gibi sabit: 32-16-8
LDIR = ART / "lofo"; (LDIR / "models").mkdir(parents=True, exist_ok=True)
FP = utils.fingerprint(cfg, ["lofo_seeds", "lofo_rf_trees", "resolution_windows", "resolution_flows_per_day",
                             "resolution_budget_per_week", "resolution_cal_windows", "resolution_eval_windows", "resolution_rho",
                             "alpha", "beta"])
FAMS = L.all_families(cfg)
RUN = utils.new_run_dir(Path(PROJECT_DIR) / cfg.runs_dir, "final_F2_lofo")
utils.save_json(utils.environment_info(), RUN / "environment.json"); cfg.save(RUN / "config_used.json")
print("katmanlar:", FAMS, "| tohumlar:", cfg.lofo_seeds, "| AE:", spec, "| parmak izi:", FP)

## 1 · LOFO tohumları için AE eğitimi (yalnız normal veri; her tohum bir kez, Drive'a kaydedilir)

In [ ]:
X0 = P0["X"]
for seed in cfg.lofo_seeds:
    mf, jf = LDIR / "models" / f"ae_seed{seed}.keras", LDIR / "models" / f"ae_seed{seed}.json"
    if RESUME and mf.exists() and jf.exists():
        print("tohum", seed, "Drive'da kayıtlı, atlanıyor (devam modu)"); continue
    utils.set_seed(seed)
    m = M.build_autoencoder(X0["benign_train"].shape[1], spec["hidden"], spec["dropout"], cfg.learning_rate)
    h = M.train_autoencoder(m, X0["benign_train"], X0["benign_val"], cfg.epochs, cfg.batch_size, cfg.patience,
                            conv_window=cfg.conv_window, conv_rel_tol=cfg.conv_rel_tol)
    m.save(mf)
    utils.save_json({"tohum": seed, "epoch": h["epochs_run"], "durma_nedeni": h["stopped_by"], "en_iyi_epoch": h["best_epoch"],
                     "en_iyi_dogrulama_kaybi": float(min(h["val_loss"]))}, jf)
    print("tohum", seed, "tamam:", h["epochs_run"], "epoch,", h["stopped_by"], flush=True)
TRS = pd.DataFrame([utils.load_json(LDIR / "models" / f"ae_seed{s}.json") for s in cfg.lofo_seeds])
TRS.to_csv(RUN / "F2_ae_egitim.csv", index=False); TRS

## 2 · Katmanlar: hazırlama, normal-veri kontrolü, host eşlemesi ve değerlendirme (her katman × tohum Drive'a kaydedilir)

In [ ]:
import json
SPL = ["benign_train", "benign_val", "benign_cal", "benign_test", "dev_test", "heldout_test"]
for fi, fam in enumerate(FAMS):
    fdir = LDIR / f"katman_{fam}"; fdir.mkdir(exist_ok=True)
    cfg_f = L.fold_config(cfg, fam)
    todo = [s for s in cfg.lofo_seeds if not (RESUME and (fdir / f"seed{s}_{FP}.json").exists())]
    if not todo:
        print(f"[{fam}] tüm tohumlar kayıtlı, atlanıyor"); continue
    if (fdir / "prepared" / "prepared.npz").exists():
        Pf = D.load_prepared(fdir / "prepared")
    else:
        print(f"[{fam}] hazırlanıyor...", flush=True)
        Pf = D.prepare(raw, cfg_f); D.save_prepared(Pf, fdir / "prepared")
        utils.save_json(Pf["report"], fdir / "prepare_report.json")
    assert L.same_benign(Pf, P0), f"{fam}: normal veri ana bölmeden farklı; AE yeniden kullanılamaz"
    hm = fdir / "host_rows.csv.gz"
    if hm.exists():
        ROWS = pd.read_csv(hm, keep_default_na=False); ROWS["split"] = ROWS["split"].replace({"": None})
    else:
        ROWS = R.map_rows(raw, Pf, cfg_f); ROWS.to_csv(hm, index=False)
    ROWS = ROWS[ROWS["split"].isin(["benign_val", "benign_test", "dev_test", "heldout_test"])].reset_index(drop=True)
    for seed in todo:
        ae = tf.keras.models.load_model(LDIR / "models" / f"ae_seed{seed}.keras", compile=False)
        S = {sp: M.ae_score(ae, Pf["X"][sp]) for sp in SPL}
        rf, thr = L.fit_rf(Pf, cfg_f, seed)
        flow = L.flow_level(Pf, S, rf, thr, cfg_f, fam, seed)
        host = L.host_level(ROWS, Pf, S, rf, thr, cfg_f, fam, seed, np.random.default_rng(10_000 + 100 * fi + seed))
        utils.save_json({"flow": flow, "host": host}, fdir / f"seed{seed}_{FP}.json")
        print(f"[{fam}] tohum {seed} tamam: AE ANOMALY {flow['AE_ANOMALY']:.3f} | RF {flow['RF_tek_recall_α']:.3f}", flush=True)
print("tüm katmanlar tamam")

## 3 · Akış düzeyi: her katmanda görülmemiş aile (tohum ortalaması ± std)

In [ ]:
FLOW, HOST = [], []
for fam in FAMS:
    for s in cfg.lofo_seeds:
        z = utils.load_json(LDIR / f"katman_{fam}" / f"seed{s}_{FP}.json")
        FLOW.append(z["flow"]); HOST += z["host"]
FLOW, HOST = pd.DataFrame(FLOW), pd.DataFrame(HOST)
FLOW.to_csv(RUN / "F2_akis_tum_tohumlar.csv", index=False); HOST.to_csv(RUN / "F2_host_tum_tohumlar.csv", index=False)
fcols = ["AE_ANOMALY", "AE_UNCERTAIN", "AE_kaçan", "AE_ANOMALY_çakışma_kaçmış", "RF_tek_recall_α", "RF_tek_recall_α+β",
         "AE+Basamak1_recall", "normal_AE_ANOMALY", "normal_RF_FPR_α", "normal_AE+Basamak1_FPR"]
FT = FLOW.groupby("aile", sort=False)[fcols].agg(["mean", "std"]).round(4)
FT.to_csv(RUN / "F2_akis_katman_ozet.csv"); FT

## 4 · Host düzeyi (Basamak 2) ve önceden kaydedilmiş başarı / başarısızlık ölçütleri

In [ ]:
rho1 = "karisik_ρ=1.00"
hcols = ["k_yildiz", "normal_haftalik_ort", "normal_haftalik_p95_host", "normal_haftalik_en_kotu_host", "saf_pencere_tespit"] + \
        [c for c in HOST.columns if c.startswith("karisik_")] + ["cikmaz_orani"]
HT = HOST.groupby(["sistem", "W", "F", "aile"], sort=False)[hcols].mean()
HT.to_csv(RUN / "F2_host_katman_ozet.csv")
fm = FLOW.groupby("aile")[["AE_ANOMALY", "RF_tek_recall_α"]].mean()
verdict = []
for (sysn, W, F), g in HOST[HOST["sistem"] == "AE + Basamak 2"].groupby(["sistem", "W", "F"]):
    gm = g.groupby("aile")[[rho1, "normal_haftalik_en_kotu_host"]].mean()
    gm = gm.join(fm)
    gm["azalma_puan"] = 100 * (gm[rho1] - gm["AE_ANOMALY"])          # kaçan+ertelenen akış − host düzeyi kaçan
    gm["bütçe_içinde"] = gm["normal_haftalik_en_kotu_host"] <= cfg.resolution_budget_per_week
    gm["katman_başarılı"] = (gm["azalma_puan"] >= 100 * cfg.lofo_success_points) & gm["bütçe_içinde"]
    gm["benzemeyen_aile"] = gm["RF_tek_recall_α"] < cfg.lofo_unlike_rf_recall
    gm["RF_yetişiyor"] = gm["RF_tek_recall_α"] >= gm[rho1] - cfg.lofo_match_margin
    n_ok = int(gm["katman_başarılı"].sum())
    fail = bool((gm["benzemeyen_aile"] & gm["RF_yetişiyor"]).any())
    verdict.append({"W": W, "F": F, "başarılı_katman": n_ok, "katman": len(gm),
                    "başarı_ölçütü_sağlandı": n_ok >= cfg.lofo_success_folds,
                    "benzemeyen_aileler": ", ".join(gm.index[gm["benzemeyen_aile"]]) or "yok",
                    "başarısızlık_ölçütü_tetiklendi": fail})
    gm.round(4).to_csv(RUN / f"F2_olcut_W{W}_F{F}.csv")
VER = pd.DataFrame(verdict); VER.to_csv(RUN / "F2_on_kayitli_olcutler.csv", index=False); VER

In [ ]:
F0 = max(cfg.resolution_flows_per_day)
fig, axs = plt.subplots(1, len(cfg.resolution_windows), figsize=(4.2 * len(cfg.resolution_windows), 4), sharey=True)
for ax, W in zip(np.atleast_1d(axs), cfg.resolution_windows):
    fams = list(fm.index); x = np.arange(len(fams)); w = 0.2
    ae = FLOW.groupby("aile")["AE_ANOMALY"].mean().reindex(fams)
    aeu = (FLOW.groupby("aile")["AE_ANOMALY"].mean() + FLOW.groupby("aile")["AE_UNCERTAIN"].mean()).reindex(fams)
    rf = FLOW.groupby("aile")["RF_tek_recall_α"].mean().reindex(fams)
    h2 = HOST[(HOST["sistem"] == "AE + Basamak 2") & (HOST["W"] == W) & (HOST["F"] == F0)].groupby("aile")[rho1].mean().reindex(fams)
    for j, (vals, lab, col) in enumerate([(ae, "AE ANOMALY (akış)", "#e34948"), (aeu, "AE ANOMALY+UNCERTAIN (akış)", "#eda100"),
                                          (rf, "yalnız RF (akış)", "#2a78d6"), (h2, f"AE + Basamak 2 (host, ρ=1)", "#1baf7a")]):
        ax.bar(x + (j - 1.5) * w, 100 * vals, w, label=lab, color=col)
    ax.set_xticks(x); ax.set_xticklabels(fams, rotation=45, ha="right", fontsize=8); ax.set_title(f"W = {W}, F = {F0}/gün")
np.atleast_1d(axs)[0].set_ylabel("görülmemiş aile tespiti (%)"); np.atleast_1d(axs)[0].legend(fontsize=7)
fig.suptitle("LOFO: her aile sırayla görülmemiş (tohum 10–19 ortalaması)"); fig.tight_layout(); fig.savefig(RUN / "F2_lofo.png", dpi=150)
print("Sonuçlar:", RUN)

In [ ]:
# Colab: Drive'a yazılan tüm dosyaların kaydedildiğinden emin ol (oturumu kapatmadan önce mutlaka çalıştırın)
print("Kaydedilen ilerleme:", utils.progress(cfg))
if IN_COLAB:
    drive.flush_and_unmount()
    print("Drive kaydedildi ve ayrıldı. Sonraki defter Drive'ı yeniden bağlar.")